In [ ]:
# 1. Import Required Libraries
import os
import itertools
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
from PIL import Image
import scipy.ndimage as ndimage
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, precision_score, recall_score, f1_score
from tensorflow.keras import layers, models

print('TensorFlow version:', tf.__version__)
print('Libraries imported successfully!')

In [ ]:
# 2. Dataset Discovery and Preprocessing
candidates = [
    Path('upload/trained data'),
    Path('../upload/trained data'),
    Path(r'C:\Project\upload\trained data'),
    Path(r'D:\Train_dataset')
]
DATASET_PATH = next((p for p in candidates if p.exists() and (p / 'no').exists() and (p / 'yes').exists()), None)
if DATASET_PATH is None:
    raise FileNotFoundError("Dataset path containing 'no' and 'yes' folders not found.")
print('Using dataset path:', DATASET_PATH)

# Class mapping: 0 for Normal ('no'), 1 for Brain Tumor ('yes')
CLASS_NAMES = ['no', 'yes']
CLASS_LABELS = {'no': 0, 'yes': 1}

IMG_SIZE = (224, 224)
SEED = 42

image_paths = []
labels = []

for class_name in CLASS_NAMES:
    folder = Path(DATASET_PATH) / class_name
    target_label = CLASS_LABELS[class_name]
    for file in folder.iterdir():
        if file.suffix.lower() in ['.jpg', '.jpeg', '.png', '.bmp']:
            image_paths.append(str(file))
            labels.append(target_label)

image_paths = np.array(image_paths)
labels = np.array(labels)

print('Total scans:', len(image_paths))
print('Normal (no, class 0):', np.sum(labels == 0))
print('Brain Tumor (yes, class 1):', np.sum(labels == 1))

In [ ]:
# 3. Stratified Train / Validation / Test Splits
X_train, X_temp, y_train, y_temp = train_test_split(
    image_paths,
    labels,
    test_size=0.30,
    random_state=SEED,
    stratify=labels
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=SEED,
    stratify=y_temp
)

print(f'Training scans:   {len(X_train)} ({len(X_train)/len(labels)*100:.1f}%)')
print(f'Validation scans: {len(X_val)} ({len(X_val)/len(labels)*100:.1f}%)')
print(f'Testing scans:    {len(X_test)} ({len(X_test)/len(labels)*100:.1f}%)')

In [ ]:
# 4. MobileNetV2 Deep Transfer Learning Feature Extractor
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet',
    pooling='avg'
)
base_model.trainable = False

def extract_features(paths, batch_size=16):
    features = []
    for i in range(0, len(paths), batch_size):
        batch_paths = paths[i:i+batch_size]
        batch_imgs = []
        for p in batch_paths:
            img = Image.open(p).convert('RGB').resize(IMG_SIZE, Image.Resampling.LANCZOS)
            arr = np.array(img, dtype=np.float32) / 255.0 * 2.0 - 1.0  # Normalized to [-1, 1]
            batch_imgs.append(arr)
        f = base_model.predict(np.array(batch_imgs, dtype=np.float32), verbose=0)
        features.append(f)
    return np.vstack(features)

print('Extracting deep features from train, val, and test scans...')
X_train_feats = extract_features(X_train)
X_val_feats = extract_features(X_val)
X_test_feats = extract_features(X_test)
print('Feature extraction completed! Vector dimensions:', X_train_feats.shape)

In [ ]:
# 5. Build and Train Classification Head
classifier = models.Sequential([
    layers.Input(shape=(1280,)),
    layers.BatchNormalization(),
    layers.Dropout(0.4),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(1, activation='sigmoid')
], name='sequential')

classifier.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

history = classifier.fit(
    X_train_feats, y_train,
    validation_data=(X_val_feats, y_val),
    epochs=20,
    batch_size=16,
    verbose=1
)

# Assemble unified end-to-end model for inference and Grad-CAM
inputs = layers.Input(shape=(224, 224, 3), name='image_input')
scaled = layers.Rescaling(scale=2.0, offset=-1.0, name='mobilenet_scaling')(inputs)
extracted = base_model(scaled, training=False)
outputs = classifier(extracted)

model = models.Model(inputs=inputs, outputs=outputs, name='Brain_Tumor_MobileNetV2')
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
print('End-to-end model assembled successfully!')
model.summary()

In [ ]:
# 6. Model Evaluation on Test Split
test_preds_raw = classifier.predict(X_test_feats, verbose=0)
test_preds = (test_preds_raw >= 0.5).astype(int).flatten()
true_labels = y_test
prediction = test_preds

print('Predicted sample:', test_preds[:10])
print('True sample:     ', true_labels[:10])

In [ ]:
# 7. Confusion Matrix Visualization
def plot_confusion_matrix(cm, classes, title='Confusion Matrix', cmap=plt.cm.Blues):
    plt.imshow(cm, interpolation='nearest', cmap=cmap)
    plt.title(title, fontsize=14)
    plt.colorbar()
    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes, fontsize=11)
    plt.yticks(tick_marks, classes, fontsize=11)
    thresh = cm.max() / 2.
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        plt.text(j, i, format(cm[i, j], 'd'),
                 horizontalalignment="center",
                 color="white" if cm[i, j] > thresh else "black",
                 fontsize=12)
    plt.tight_layout()
    plt.ylabel('True Class', fontsize=12)
    plt.xlabel('Predicted Class', fontsize=12)

cm = confusion_matrix(y_true=true_labels, y_pred=prediction)
cm_plot_labels = ['Normal', 'Brain Tumor']

plt.figure(figsize=(6, 5))
plot_confusion_matrix(cm=cm, classes=cm_plot_labels)
plt.show()

In [ ]:
# 8. Accuracy & Performance Metrics Verification
acc = accuracy_score(true_labels, prediction)
precision = precision_score(true_labels, prediction, zero_division=0)
recall = recall_score(true_labels, prediction, zero_division=0)
score = f1_score(true_labels, prediction, zero_division=0)

print('========================================')
print(f'Test Accuracy:  {acc:.4f} ({acc*100:.2f}%)')
print(f'Test Precision: {precision:.4f} ({precision*100:.2f}%)')
print(f'Test Recall:    {recall:.4f} ({recall*100:.2f}%)')
print(f'Test F1-Score:  {score:.4f} ({score*100:.2f}%)')
print('========================================')
print('\n--- Classification Report ---')
print(classification_report(true_labels, prediction, target_names=cm_plot_labels))

# Save model to disk (compact 11.18 MB)
MODEL_SAVE_PATH = 'best_brain_tumor_model.keras'
model.save(MODEL_SAVE_PATH)
print(f'Model saved successfully to: {MODEL_SAVE_PATH}')

# Update Accuracy.txt
with open('Accuracy.txt', 'w') as f:
    f.write(f'CNN (MobileNetV2 Transfer Learning Architecture)\n\nAccuracy score:  {acc:.4f}\n')
print('Accuracy.txt updated successfully!')

## 9. Explainable AI: Grad-CAM Brain Tumor Localization in Red

To provide clinical interpretability, **Gradient-weighted Class Activation Mapping (Grad-CAM)** backpropagates gradients from the un-saturated tumor logit into the MobileNetV2 `out_relu` convolutional layer.

The focal tumor area is showcased in **vivid red** with an automated boundary contour, allowing clinicians to instantly localize and verify the tumor mass.

In [ ]:
# 10. Grad-CAM Red Tumor Localization Function
def compute_gradcam(image_path, model_path='best_brain_tumor_model.keras'):
    """
    Compute Grad-CAM activation heatmap and generate a Red Tumor Localization Overlay.
    """
    loaded_model = tf.keras.models.load_model(model_path)
    rescaling = loaded_model.get_layer('mobilenet_scaling')
    base = loaded_model.get_layer('mobilenetv2_1.00_224')
    pool = base.get_layer('global_average_pooling2d')
    clf = loaded_model.get_layer('sequential')
    last_conv_layer = base.get_layer('out_relu')
    last_conv_model = tf.keras.models.Model(inputs=base.inputs, outputs=last_conv_layer.output)
    
    dense_2 = clf.layers[-1]
    w, b = dense_2.get_weights()

    # Load and preprocess image
    orig_img = Image.open(image_path).convert('RGB')
    w_orig, h_orig = orig_img.size
    img_resized = orig_img.resize((224, 224), Image.Resampling.LANCZOS)
    img_arr = np.expand_dims(np.array(img_resized, dtype=np.float32) / 255.0, 0)
    scaled_input = rescaling(img_arr)

    # Forward pass through feature extractor outside tape
    conv_outputs = last_conv_model(scaled_input)

    # Compute gradients with respect to un-saturated tumor logit
    with tf.GradientTape() as tape:
        tape.watch(conv_outputs)
        pooled = pool(conv_outputs)
        h = pooled
        for l in clf.layers[:-1]:
            h = l(h, training=False)
        logit = tf.matmul(h, w) + b
        loss = logit[0, 0]

    grads = tape.gradient(loss, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    cam = tf.squeeze(conv_outputs[0] @ pooled_grads[..., tf.newaxis])
    cam = tf.maximum(cam, 0.0)
    max_c = tf.reduce_max(cam)
    if max_c > 0:
        cam = cam / max_c
    cam_np = cam.numpy()

    # Smoothly upscale heatmap to original scan dimensions
    cam_img = Image.fromarray(np.uint8(255 * cam_np)).resize((w_orig, h_orig), Image.Resampling.BICUBIC)
    cam_smooth = np.array(cam_img, dtype=np.float32) / 255.0
    orig_np = np.array(orig_img, dtype=np.float32)

    # 1. Red Tumor Highlight Overlay (showcases tumor in vivid red)
    alpha = np.clip((cam_smooth - 0.25) / 0.75, 0.0, 1.0)[:, :, np.newaxis]
    alpha = (alpha ** 1.3) * 0.75

    red_tint = np.zeros_like(orig_np, dtype=np.float32)
    red_tint[:, :, 0] = 255.0  # Vivid red channel
    red_tint[:, :, 1] = 25.0
    red_tint[:, :, 2] = 25.0

    red_overlay = (1.0 - alpha) * orig_np + alpha * red_tint

    # Add crisp boundary outline around tumor core (>= 0.48)
    high_mask = cam_smooth >= 0.48
    if np.any(high_mask):
        dilated = ndimage.binary_dilation(high_mask, iterations=3)
        boundary = dilated & (~high_mask)
        red_overlay[boundary, 0] = 255.0
        red_overlay[boundary, 1] = 30.0
        red_overlay[boundary, 2] = 30.0

    red_overlay_uint8 = np.clip(red_overlay, 0, 255).astype(np.uint8)

    # 2. Thermal Jet Heatmap Overlay
    jet = matplotlib.colormaps['jet']
    jet_colors = jet(cam_smooth)[:, :, :3] * 255.0
    jet_overlay = np.clip(0.52 * orig_np + 0.48 * jet_colors, 0, 255).astype(np.uint8)

    prob = float(tf.sigmoid(logit).numpy()[0, 0])
    return orig_np.astype(np.uint8), red_overlay_uint8, jet_overlay, prob

print('compute_gradcam defined successfully!')

In [ ]:
# 11. Demonstrate Grad-CAM Red Tumor Localization on Test Scans
sample_scans = [
    'upload/Brain Tumor/image.png',
    'upload/Brain Tumor/image2.png',
    'upload/Normal/image.png'
]

fig, axes = plt.subplots(len(sample_scans), 3, figsize=(15, 5 * len(sample_scans)))

for idx, scan_path in enumerate(sample_scans):
    orig, red_cam, jet_cam, prob = compute_gradcam(scan_path)
    label = 'Brain Tumor' if prob >= 0.5 else 'Normal'
    conf_str = f'{prob*100.0:.1f}%'
    
    axes[idx, 0].imshow(orig)
    axes[idx, 0].set_title(f'Original Scan\n{Path(scan_path).name}', fontsize=11)
    axes[idx, 0].axis('off')

    axes[idx, 1].imshow(red_cam)
    axes[idx, 1].set_title(f'Grad-CAM: Tumor Area (Red)\n{label} ({conf_str})', fontsize=11, color='red' if prob >= 0.5 else 'green')
    axes[idx, 1].axis('off')

    axes[idx, 2].imshow(jet_cam)
    axes[idx, 2].set_title('Thermal Heatmap Spectrum', fontsize=11)
    axes[idx, 2].axis('off')

plt.tight_layout()
plt.show()
print('Grad-CAM Red Tumor Localization successfully demonstrated!')